# Lung Segmentation and Disease Classification from Chest X-Rays
**Objective:** Train a U-Net model to segment the lungs from X-ray images.

**Team Members:** 
Khaled El-Sharkawy, Ahmed Bassem , Ahmed Adel , Yousef Usama, Abdulrahman Ahmed.

**Dataset:** COVID-19 Radiography Database.

In [1]:
import os
import cv2
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, LeakyReLU, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
normal_images_dir = '/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images' # Set the directory path for healthy chest X-ray images
normal_masks_dir = '/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks' # Set the directory path for the corresponding lung masks

image_paths = [] # Initialize an empty list to store valid image paths
mask_paths = [] # Initialize an empty list to store valid mask paths
filenames = os.listdir(normal_images_dir) # Retrieve a list of all filenames in the images directory

for filename in filenames[:1500]: # Iterate over the first 1500 files to prevent out-of-memory errors
    img_path = os.path.join(normal_images_dir, filename) # Combine the directory and filename to create the full image path
    mask_path = os.path.join(normal_masks_dir, filename) # Combine the directory and filename to create the full mask path
    if os.path.exists(mask_path): # Check if the corresponding mask file actually exists on the disk
        image_paths.append(img_path) # Append the verified image path to the list
        mask_paths.append(mask_path) # Append the verified mask path to the list

print(f"Total valid image-mask pairs found: {len(image_paths)}") # Print the total count of successfully matched image-mask pairs

img_size = 256 # Define the target spatial dimensions for the images and masks

def load_data(img_paths, mask_paths): # Define a custom function to read, resize, and normalize the data
    images, masks = [], [] # Initialize empty lists to hold the processed image and mask arrays
    for i in range(len(img_paths)): # Start a loop to iterate through all provided paths
        img = cv2.imread(img_paths[i], cv2.IMREAD_GRAYSCALE) # Read the original image in grayscale mode
        img = cv2.resize(img, (img_size, img_size)) / 255.0 # Resize the image to 256x256 and normalize pixel values to range between 0 and 1
        img = np.expand_dims(img, axis=-1) # Add a channel dimension required by the convolutional layers
        images.append(img) # Append the fully processed image array to the list
        
        mask = cv2.imread(mask_paths[i], cv2.IMREAD_GRAYSCALE) # Read the corresponding mask in grayscale mode
        mask = cv2.resize(mask, (img_size, img_size)) / 255.0 # Resize and normalize the mask pixels
        mask = (mask > 0.5).astype(np.float32) # Apply a threshold to convert the mask into strictly binary values
        mask = np.expand_dims(mask, axis=-1) # Add the final channel dimension to the mask array
        masks.append(mask) # Append the fully processed mask array to the list
        
    return np.array(images), np.array(masks) # Convert the final lists to NumPy arrays and return them

X, y = load_data(image_paths, mask_paths) # Execute the data loading function on the full path lists
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42) # Split the dataset into 80 percent training and 20 percent validation subsets

Total valid image-mask pairs found: 1500


In [3]:
def unet_model(input_size=(256, 256, 1)): # Define a function to build the U-Net architecture and specify the input shape
    inputs = Input(input_size) # Create the primary input layer with the specified spatial dimensions
    
    conv1 = Conv2D(32, 3, padding='same')(inputs) # Apply the first convolutional layer with 32 filters while preserving spatial dimensions
    conv1 = LeakyReLU(alpha=0.1)(conv1) # Apply the LeakyReLU activation function to prevent the dying ReLU problem
    conv1 = Conv2D(32, 3, padding='same')(conv1) # Apply a second convolutional layer with 32 filters
    conv1 = LeakyReLU(alpha=0.1)(conv1) # Apply the LeakyReLU activation function again
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1) # Reduce spatial dimensions by half using max pooling
    
    conv2 = Conv2D(64, 3, padding='same')(pool1) # Apply a convolutional layer with 64 filters
    conv2 = LeakyReLU(alpha=0.1)(conv2) # Apply activation
    conv2 = Conv2D(64, 3, padding='same')(conv2) # Apply another convolutional layer with 64 filters
    conv2 = LeakyReLU(alpha=0.1)(conv2) # Apply activation
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2) # Perform a second downsampling operation
    
    conv3 = Conv2D(128, 3, padding='same')(pool2) # Apply a convolutional layer with 128 filters at the network bottleneck
    conv3 = LeakyReLU(alpha=0.1)(conv3) # Apply activation
    conv3 = Conv2D(128, 3, padding='same')(conv3) # Apply a second 128-filter convolutional layer
    conv3 = LeakyReLU(alpha=0.1)(conv3) # Apply activation
    
    up4 = UpSampling2D(size=(2, 2))(conv3) # Double the spatial dimensions to begin the expansive path
    up4 = concatenate([up4, conv2], axis=3) # Merge the upsampled features with the corresponding skip connection from the encoder
    conv4 = Conv2D(64, 3, padding='same')(up4) # Apply a convolutional layer with 64 filters
    conv4 = LeakyReLU(alpha=0.1)(conv4) # Apply activation
    conv4 = Conv2D(64, 3, padding='same')(conv4) # Apply another convolutional layer with 64 filters
    conv4 = LeakyReLU(alpha=0.1)(conv4) # Apply activation
    
    up5 = UpSampling2D(size=(2, 2))(conv4) # Perform the final upsampling operation
    up5 = concatenate([up5, conv1], axis=3) # Merge with the high-resolution features from the first convolutional block
    conv5 = Conv2D(32, 3, padding='same')(up5) # Apply a convolutional layer with 32 filters
    conv5 = LeakyReLU(alpha=0.1)(conv5) # Apply activation
    conv5 = Conv2D(32, 3, padding='same')(conv5) # Apply another convolutional layer with 32 filters
    conv5 = LeakyReLU(alpha=0.1)(conv5) # Apply activation
    
    outputs = Conv2D(1, 1, activation='sigmoid')(conv5) # Apply a 1x1 convolution with a sigmoid activation to output pixel-wise probabilities
    return Model(inputs=inputs, outputs=outputs) # Encapsulate the layers into a Keras Model instance and return it

def dice_coef(y_true, y_pred, smooth=1e-6): # Define a custom metric function to calculate the Dice Coefficient
    y_true_f = tf.keras.backend.flatten(y_true) # Flatten the ground truth mask into a 1D vector
    y_pred_f = tf.keras.backend.flatten(y_pred) # Flatten the predicted mask into a 1D vector
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f) # Calculate the intersection of the two masks
    return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth) # Return the computed Dice score

model = unet_model() # Instantiate the U-Net architecture
model.compile(optimizer=Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=[dice_coef, 'accuracy']) # Compile the model with the Adam optimizer and binary crossentropy loss

checkpoint = ModelCheckpoint('unet_lung_model.keras', monitor='val_dice_coef', mode='max', save_best_only=True, verbose=1) # Configure a callback to save the model weights that achieve the highest validation Dice score
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True) # Configure a callback to halt training if validation loss does not improve for 5 epochs

history = model.fit( X_train, y_train, validation_data=(X_val, y_val), epochs=20, batch_size=16, callbacks=[checkpoint, early_stop]) # Execute the training loop using the compiled model and data

del X, y, X_train, X_val, y_train, y_val, model # Delete large arrays and the model instance from memory
gc.collect() # Force the garbage collector to release the unreferenced memory back to the system

I0000 00:00:1788292838.275458      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1788292838.278399      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 1/20


2026-09-01 20:00:56.039146: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-01 20:00:56.350560: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-01 20:00:58.142577: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-01 20:00:58.490434: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng4{k11=1} for conv (f32[16,192,128,128]{3,2,1,0}, u8[0]{0}) custom-call(f32[16,64,128,128]{3,2,1,0}, f32[64,192,3,3]{3,2,1,0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, 

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step - accuracy: 0.6528 - dice_coef: 0.3115 - loss: 0.6237
Epoch 1: val_dice_coef improved from None to 0.28943, saving model to unet_lung_model.keras

Epoch 1: finished saving model to unet_lung_model.keras
75/75 ━━━━━━━━━━━━━━━━━━━━ 52s 305ms/step - accuracy: 0.7244 - dice_coef: 0.2964 - loss: 0.5713 - val_accuracy: 0.7485 - val_dice_coef: 0.2894 - val_loss: 0.5276
Epoch 2/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 219ms/step - accuracy: 0.7551 - dice_coef: 0.3091 - loss: 0.4962
Epoch 2: val_dice_coef improved from 0.28943 to 0.56204, saving model to unet_lung_model.keras

Epoch 2: finished saving model to unet_lung_model.keras
75/75 ━━━━━━━━━━━━━━━━━━━━ 18s 237ms/step - accuracy: 0.7803 - dice_coef: 0.3662 - loss: 0.4429 - val_accuracy: 0.8783 - val_dice_coef: 0.5620 - val_loss: 0.3241
Epoch 3/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 229ms/step - accuracy: 0.8844 - dice_coef: 0.6326 - loss: 0.2916
Epoch 3: val_dice_coef improved from 0.56204 to 0.70109, saving mode

816

In [4]:
base_dir = '/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset' # Define the root directory path for the classification dataset
categories = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia'] # Define class names in strict alphabetical order to match the Streamlit interface mapping exactly
X_cls, y_cls = [], [] # Initialize empty lists to hold the segmented lung images and their labels
IMG_SIZE_CLS = 224 # Set the target image resolution required by the DenseNet121 architecture

print("Loading dataset... (Extracting 1200 images per class for high accuracy)") # Display a status message indicating the start of data loading

for label, category in enumerate(categories): # Iterate over each category and generate a numerical label corresponding to its alphabetical position
    cat_images_dir = os.path.join(base_dir, category, 'images') # Construct the directory path for the current category raw images
    cat_masks_dir = os.path.join(base_dir, category, 'masks') # Construct the directory path for the current category masks
    filenames = os.listdir(cat_images_dir)[:1200] # Retrieve the first 1200 files to ensure a balanced dataset across classes
    
    for filename in filenames: # Iterate through the selected filenames
        img_path = os.path.join(cat_images_dir, filename) # Construct the full path to the specific image
        mask_path = os.path.join(cat_masks_dir, filename) # Construct the full path to the specific mask
        
        if os.path.exists(mask_path): # Ensure the mask exists to prevent file not found errors
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE) # Read the original image in grayscale
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) # Read the corresponding mask in grayscale
            img = cv2.resize(img, (IMG_SIZE_CLS, IMG_SIZE_CLS)) # Resize the image to 224x224
            mask = cv2.resize(mask, (IMG_SIZE_CLS, IMG_SIZE_CLS)) # Resize the mask to 224x224
            
            mask_bin = (mask > 127).astype(np.uint8) # Binarize the mask to isolate the lung area completely
            segmented_lung = img * mask_bin # Multiply the original image by the binary mask to remove the background
            segmented_lung_3c = cv2.cvtColor(segmented_lung, cv2.COLOR_GRAY2RGB) / 255.0 # Convert the segmented grayscale image to RGB and normalize the pixel values
            
            X_cls.append(segmented_lung_3c) # Append the finalized RGB segmented lung image to the dataset list
            y_cls.append(label) # Append the corresponding integer label

X_cls = np.array(X_cls) # Convert the list of images into a multi-dimensional NumPy array
y_cls = to_categorical(np.array(y_cls), num_classes=4) # Apply one-hot encoding to the integer labels for categorical crossentropy
X_train_cls, X_val_cls, y_train_cls, y_val_cls = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42) # Split the dataset into 80 percent training and 20 percent validation sets

print(f"Data Ready! Total Training Samples: {X_train_cls.shape[0]}") # Output a confirmation message with the total number of training samples
del X_cls, y_cls # Delete the massive original arrays to free up system memory
gc.collect() # Trigger the garbage collector immediately

Loading dataset... (Extracting 1200 images per class for high accuracy)
Data Ready! Total Training Samples: 3840


0

In [5]:
aug = ImageDataGenerator( # Initialize the ImageDataGenerator to perform real-time data augmentation
    rotation_range=10, # Randomly rotate the images by up to 10 degrees during training
    zoom_range=0.1, # Randomly apply a 10 percent zoom in or out
    width_shift_range=0.1, # Randomly shift the image horizontally by 10 percent of its total width
    height_shift_range=0.1, # Randomly shift the image vertically by 10 percent of its total height
    horizontal_flip=False # Explicitly disable horizontal flipping to maintain correct anatomical heart placement
) # Finalize the augmentation configuration

base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(IMG_SIZE_CLS, IMG_SIZE_CLS, 3)) # Load the DenseNet121 architecture pre-trained on ImageNet, excluding its final classification layers
base_model.trainable = True # Unfreeze the base model layers to allow fine-tuning on our specific medical dataset

inputs_cls = Input(shape=(IMG_SIZE_CLS, IMG_SIZE_CLS, 3)) # Define the exact shape for the input tensor
x = base_model(inputs_cls) # Pass the input tensor through the pre-trained DenseNet feature extractor
x = GlobalAveragePooling2D()(x) # Apply global average pooling to flatten the 2D feature maps into a 1D vector
x = Dense(256, activation='relu')(x) # Add a fully connected dense layer with 256 units and ReLU activation
x = Dropout(0.5)(x) # Apply a 50 percent dropout rate to reduce overfitting during training
outputs_cls = Dense(4, activation='softmax')(x) # Add the final output layer with 4 units and softmax activation for multi-class probabilities

cls_model = Model(inputs_cls, outputs_cls) # Construct the final classification model connecting inputs to outputs

cls_model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy']) # Compile the model using Adam optimizer and categorical crossentropy loss function

rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1, min_lr=1e-6) # Configure a callback to halve the learning rate if the validation loss plateaus for 2 epochs
checkpoint_cls = ModelCheckpoint('medical_classifier_densenet.keras', monitor='val_accuracy', mode='max', save_best_only=True, verbose=1) # Configure a callback to save the best model weights based on validation accuracy
early_stop_cls = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True) # Configure a callback to stop training and restore best weights if validation loss does not improve for 6 epochs

history_cls = cls_model.fit( # Start the model training process
    aug.flow(X_train_cls, y_train_cls, batch_size=32), # Feed the training data through the augmentation generator in batches of 32
    validation_data=(X_val_cls, y_val_cls), # Provide the validation data to evaluate the model after each epoch
    epochs=20, # Set the maximum number of training cycles over the entire dataset
    callbacks=[checkpoint_cls, early_stop_cls, rlr] # Pass the list of callbacks to monitor and control the training process
) # Conclude the training setup

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 334ms/step - accuracy: 0.5960 - loss: 0.9979
Epoch 1: val_accuracy improved from None to 0.68854, saving model to medical_classifier_densenet.keras

Epoch 1: finished saving model to medical_classifier_densenet.keras
120/120 ━━━━━━━━━━━━━━━━━━━━ 231s 482ms/step - accuracy: 0.6943 - loss: 0.7623 - val_accuracy: 0.6885 - val_loss: 0.7383 - learning_rate: 1.0000e-04
Epoch 2/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 335ms/step - accuracy: 0.8028 - loss: 0.5046
Epoch 2: val_accuracy improved from 0.68854 to 0.76354, saving model to medical_classifier_densenet.keras

Epoch 2: finished saving model to medical_classifier_densenet.keras
120/120 ━━━━━━━━━━━━━━━━━━━━ 46s 379ms/step - accuracy: 0.8026 - loss: 0.5010 - val_accuracy: 0.7635 - val_loss: 0.6489 - learning_rate: 1.0000e-04
Epoch 3/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 323ms/step - accuracy: 0.8335 - loss: 0.4247
Epoch 3: val_accuracy improved from 0.763